# Geographic Supply–Demand Gap Analysis

In [ ]:
# 1. Imports and settings

import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

print("Ready.")


In [ ]:
# 2. Load master table

# Keep master_table.csv in the same folder as this notebook.
FILE_PATH = "master_table.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"{FILE_PATH} was not found. Put the CSV beside this notebook "
        "or change FILE_PATH to the correct path."
    )

df = pd.read_csv(FILE_PATH, low_memory=False)

print("Shape:", df.shape)
print("Unique orders:", df["order_id"].nunique())
print("Unique customers:", df["customer_unique_id"].nunique())
print("Unique sellers:", df["seller_id"].nunique())
print("Unique products:", df["product_id"].nunique())


In [ ]:
# 3. Basic data-quality checks for the fields used here

geo_cols = [
    "customer_unique_id", "customer_city", "customer_state",
    "seller_id", "seller_city", "seller_state",
    "product_category_name_english"
]

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

print("Missing values in key geographic/category fields:")
quality = pd.DataFrame({
    "missing": df[geo_cols].isna().sum(),
    "missing_pct": (df[geo_cols].isna().mean() * 100).round(2)
})
print(quality)

for c in date_cols:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

print("\nDate columns converted.")


## A. Customer geographical concentration

In [ ]:
# 4. Customers by state

customer_state = (
    df.groupby("customer_state")["customer_unique_id"]
      .nunique()
      .rename("customers")
      .reset_index()
      .sort_values("customers", ascending=False)
)

customer_state["customer_share_pct"] = (
    customer_state["customers"] / customer_state["customers"].sum() * 100
).round(2)

print(customer_state.to_string(index=False))


In [ ]:
# 5. Customers by city

customer_city = (
    df.groupby(["customer_state", "customer_city"])["customer_unique_id"]
      .nunique()
      .rename("customers")
      .reset_index()
      .sort_values("customers", ascending=False)
)

customer_city["customer_share_pct"] = (
    customer_city["customers"] / customer_city["customers"].sum() * 100
).round(2)

print("Top 50 customer cities:")
print(customer_city.head(50).to_string(index=False))


## B. Seller geographical concentration

In [ ]:
# 6. Sellers by state

seller_state = (
    df.groupby("seller_state")["seller_id"]
      .nunique()
      .rename("sellers")
      .reset_index()
      .sort_values("sellers", ascending=False)
)

seller_state["seller_share_pct"] = (
    seller_state["sellers"] / seller_state["sellers"].sum() * 100
).round(2)

print(seller_state.to_string(index=False))


In [ ]:
# 7. Sellers by city

seller_city = (
    df.groupby(["seller_state", "seller_city"])["seller_id"]
      .nunique()
      .rename("sellers")
      .reset_index()
      .sort_values("sellers", ascending=False)
)

seller_city["seller_share_pct"] = (
    seller_city["sellers"] / seller_city["sellers"].sum() * 100
).round(2)

print("Top 50 seller cities:")
print(seller_city.head(50).to_string(index=False))


## C. Exact customer–seller geographic imbalance


In [ ]:
# 8. State-level customer/seller imbalance

customer_state_tmp = customer_state.rename(columns={"customer_state": "state"})
seller_state_tmp = seller_state.rename(columns={"seller_state": "state"})

geo_balance = customer_state_tmp.merge(
    seller_state_tmp,
    on="state",
    how="outer"
)

geo_balance[["customers", "sellers"]] = geo_balance[["customers", "sellers"]].fillna(0)

geo_balance["customers_per_seller"] = np.where(
    geo_balance["sellers"] > 0,
    geo_balance["customers"] / geo_balance["sellers"],
    np.inf
)

geo_balance["sellers_per_1000_customers"] = np.where(
    geo_balance["customers"] > 0,
    geo_balance["sellers"] / geo_balance["customers"] * 1000,
    np.nan
)

geo_balance["share_gap_pp"] = (
    geo_balance["seller_share_pct"] -
    geo_balance["customer_share_pct"]
).round(2)

geo_balance["imbalance"] = np.select(
    [
        geo_balance["share_gap_pp"] <= -2,
        geo_balance["share_gap_pp"] >= 2
    ],
    [
        "Customer-heavy",
        "Seller-heavy"
    ],
    default="Relatively balanced"
)

print(
    geo_balance
    .sort_values("customers_per_seller", ascending=False)
    .round(2)
    .to_string(index=False)
)


In [ ]:
# 9. Highest-priority geographic imbalance — large customer base + relatively few sellers

# Avoid flagging tiny states just because their ratio is high.
# Thresholds are transparent and can be changed.
priority_states = geo_balance[
    (geo_balance["customers"] >= geo_balance["customers"].quantile(0.50)) &
    (geo_balance["sellers"] > 0)
].copy()

priority_states["customer_to_seller_index"] = (
    priority_states["customers_per_seller"] /
    geo_balance["customers_per_seller"].replace([np.inf], np.nan).median()
)

print(
    priority_states
    .sort_values("customers_per_seller", ascending=False)
    .round(2)
    .to_string(index=False)
)


## D. Category-level seller gaps by customer region


In [ ]:
# 10. Category demand in each customer state

category_col = "product_category_name_english"

category_demand = (
    df.dropna(subset=[category_col])
      .groupby(["customer_state", category_col])["order_id"]
      .nunique()
      .rename("category_orders")
      .reset_index()
)

print("Largest customer-state/category demand combinations:")
print(
    category_demand
    .sort_values("category_orders", ascending=False)
    .head(50)
    .to_string(index=False)
)


In [ ]:
# 11. Local category seller supply

local_category_supply = (
    df.dropna(subset=[category_col])
      .groupby(["seller_state", category_col])["seller_id"]
      .nunique()
      .rename("local_category_sellers")
      .reset_index()
      .rename(columns={"seller_state": "customer_state"})
)

state_category = category_demand.merge(
    local_category_supply,
    on=["customer_state", category_col],
    how="left"
)

state_category["local_category_sellers"] = (
    state_category["local_category_sellers"].fillna(0).astype(int)
)

state_category["orders_per_local_seller"] = np.where(
    state_category["local_category_sellers"] > 0,
    state_category["category_orders"] /
    state_category["local_category_sellers"],
    np.inf
)

print("High-demand categories with few/no local sellers:")
print(
    state_category[
        state_category["category_orders"] >= 50
    ]
    .sort_values(
        ["local_category_sellers", "category_orders"],
        ascending=[True, False]
    )
    .head(100)
    .to_string(index=False)
)


In [ ]:
# 12. Order-level delivery and review metrics

order_keep = [
    "order_id",
    "customer_state",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "review_scores"
]

order_keep = [c for c in order_keep if c in df.columns]

orders = df[order_keep].drop_duplicates("order_id").copy()

orders["review_score"] = pd.to_numeric(
    orders["review_scores"], errors="coerce"
)

orders["delivery_days"] = (
    orders["order_delivered_customer_date"] -
    orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders["delivery_deviation_days"] = (
    orders["order_delivered_customer_date"] -
    orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

# Only classify delivery status when both actual and estimated dates exist.
orders["delivery_status"] = np.select(
    [
        orders["delivery_deviation_days"].notna() &
        (orders["delivery_deviation_days"] < 0),

        orders["delivery_deviation_days"].notna() &
        (orders["delivery_deviation_days"] == 0),

        orders["delivery_deviation_days"].notna() &
        (orders["delivery_deviation_days"] > 0)
    ],
    ["Early", "On time", "Late"],
    default="Unknown"
)

orders["is_late"] = np.where(
    orders["delivery_deviation_days"].notna(),
    orders["delivery_deviation_days"] > 0,
    np.nan
)

print("Order-level table:", orders.shape)
print(orders[[
    "delivery_days",
    "delivery_deviation_days",
    "review_score"
]].describe().round(2))


In [ ]:
# 13. Attach delivery/review outcomes to state-category combinations

item_base = df[[
    "order_id",
    "seller_state",
    "customer_state",
    category_col,
    "seller_id"
]].copy()

# Each order may have multiple items/categories. We deliberately keep item/category level
# here because category is the variable whose supply gap we are testing.
item_base = item_base.drop_duplicates()

state_category_orders = item_base.merge(
    orders[[
        "order_id",
        "delivery_days",
        "delivery_deviation_days",
        "delivery_status",
        "is_late",
        "review_score"
    ]],
    on="order_id",
    how="left"
)

state_category_orders["same_state"] = (
    state_category_orders["seller_state"] ==
    state_category_orders["customer_state"]
)

outcomes = (
    state_category_orders
    .groupby(["customer_state", category_col], dropna=False)
    .agg(
        orders=("order_id", "nunique"),
        same_state_orders=("same_state", "sum"),
        avg_delivery_days=("delivery_days", "mean"),
        median_delivery_days=("delivery_days", "median"),
        late_rate=("is_late", "mean"),
        avg_delay_days=("delivery_deviation_days", "mean"),
        avg_review=("review_score", "mean"),
        low_rating_rate=("review_score", lambda x: (x <= 2).mean())
    )
    .reset_index()
)

outcomes["same_state_pct"] = (
    outcomes["same_state_orders"] /
    outcomes["orders"] * 100
)

outcomes["out_of_state_pct"] = 100 - outcomes["same_state_pct"]
outcomes["late_rate_pct"] = outcomes["late_rate"] * 100
outcomes["low_rating_pct"] = outcomes["low_rating_rate"] * 100

gap_analysis = state_category.merge(
    outcomes,
    on=["customer_state", category_col],
    how="left"
)

print("State/category supply + delivery + review analysis:")
print(
    gap_analysis
    .sort_values("category_orders", ascending=False)
    .head(100)
    .round(3)
    .to_string(index=False)
)


## E. Do low-supply areas actually have worse delivery and reviews?


In [ ]:
# 14. Seller coverage buckets versus delivery/review outcomes

coverage = gap_analysis[
    gap_analysis["category_orders"] >= 50
].copy()

coverage["seller_coverage_group"] = pd.cut(
    coverage["local_category_sellers"],
    bins=[-1, 0, 1, 3, np.inf],
    labels=["0 sellers", "1 seller", "2-3 sellers", "4+ sellers"]
)

coverage_summary = (
    coverage
    .groupby("seller_coverage_group", observed=False)
    .agg(
        state_category_combinations=("customer_state", "size"),
        category_orders=("category_orders", "sum"),
        avg_delivery_days=("avg_delivery_days", "mean"),
        avg_late_rate_pct=("late_rate_pct", "mean"),
        avg_delay_days=("avg_delay_days", "mean"),
        avg_review=("avg_review", "mean"),
        avg_low_rating_pct=("low_rating_pct", "mean"),
        avg_out_of_state_pct=("out_of_state_pct", "mean")
    )
)

print(coverage_summary.round(2).to_string())


In [ ]:
# 15. Candidate gaps: high demand + low local supply + weak outcomes

# Transparent screening criteria.
# Change these thresholds if your project team chooses different definitions.
candidate_gaps = gap_analysis[
    (gap_analysis["category_orders"] >= 50) &
    (gap_analysis["local_category_sellers"] <= 1)
].copy()

candidate_gaps["priority_reason"] = np.select(
    [
        (candidate_gaps["local_category_sellers"] == 0) &
        (candidate_gaps["late_rate"] >= 0.25) &
        (candidate_gaps["avg_review"] <= 3.5),

        (candidate_gaps["local_category_sellers"] == 0),

        (candidate_gaps["local_category_sellers"] == 1) &
        (candidate_gaps["late_rate"] >= 0.25)
    ],
    [
        "No local seller + high late rate + poor reviews",
        "No local seller",
        "One local seller + high late rate"
    ],
    default="Low local coverage"
)

candidate_gaps = candidate_gaps.sort_values(
    ["category_orders", "late_rate"],
    ascending=[False, False]
)

print(
    candidate_gaps[[
        "customer_state",
        category_col,
        "category_orders",
        "local_category_sellers",
        "orders_per_local_seller",
        "same_state_pct",
        "out_of_state_pct",
        "avg_delivery_days",
        "late_rate_pct",
        "avg_delay_days",
        "avg_review",
        "low_rating_pct",
        "priority_reason"
    ]]
    .head(100)
    .round(2)
    .to_string(index=False)
)


## F. Seller → customer state routes


In [ ]:
# 16. Route-level delivery performance

route = (
    state_category_orders
    .groupby(["seller_state", "customer_state"], dropna=False)
    .agg(
        orders=("order_id", "nunique"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_delay_days=("delivery_deviation_days", "mean"),
        late_rate=("is_late", "mean"),
        avg_review=("review_score", "mean")
    )
    .reset_index()
)

route["late_rate_pct"] = route["late_rate"] * 100

print("Routes with at least 100 orders and highest late rates:")
print(
    route[
        route["orders"] >= 100
    ]
    .sort_values(["late_rate", "orders"], ascending=[False, False])
    .head(50)
    .round(2)
    .to_string(index=False)
)


In [ ]:
# 17. Same-state versus cross-state performance

same_state_summary = (
    state_category_orders
    .groupby("same_state")
    .agg(
        orders=("order_id", "nunique"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_delay_days=("delivery_deviation_days", "mean"),
        late_rate=("is_late", "mean"),
        avg_review=("review_score", "mean"),
        low_rating_rate=("review_score", lambda x: (x <= 2).mean())
    )
)

same_state_summary["late_rate_pct"] = same_state_summary["late_rate"] * 100
same_state_summary["low_rating_pct"] = same_state_summary["low_rating_rate"] * 100

print(same_state_summary.round(3).to_string())


## G. Which MQLs could be approached?

In [ ]:
# 18. Inspect available MQL/closed-deal fields

mql_cols = [
    c for c in [
        "mql_id",
        "seller_id",
        "seller_state",
        "seller_city",
        category_col,
        "origin",
        "landing_page_id",
        "lead_type",
        "lead_behaviour_profile",
        "business_segment",
        "business_type",
        "has_company",
        "has_gtin",
        "average_stock",
        "declared_product_catalog_size",
        "declared_monthly_revenue",
        "won_date",
        "first_contact_date"
    ]
    if c in df.columns
]

print("MQL-related fields found:")
print(mql_cols)


In [ ]:
# 19. Create one row per MQL

if "mql_id" not in df.columns:
    print("mql_id is not present in the master table. MQL-specific prospect matching cannot be performed.")
else:
    mql = df[mql_cols].drop_duplicates("mql_id").copy()

    print("Unique MQLs:", mql["mql_id"].nunique())

    if "won_date" in mql.columns:
        mql["closed_deal"] = mql["won_date"].notna()

    print(mql.head())


In [ ]:
# 20. If MQLs have seller geography/category, match them to identified gaps

match_fields = [
    c for c in ["seller_state", "seller_city", category_col]
    if c in mql.columns
]

if len(match_fields) == 0:
    print(
        "No seller geography or product-category field is available in the MQL data. "
        "Individual MQL-to-gap matching cannot be justified from this dataset."
    )
else:
    print("Potential MQL matching fields:", match_fields)

    if "seller_state" in mql.columns and category_col in mql.columns:
        gap_keys = candidate_gaps[
            ["customer_state", category_col]
        ].drop_duplicates()

        mql_candidates = mql.merge(
            gap_keys,
            left_on=["seller_state", category_col],
            right_on=["customer_state", category_col],
            how="inner"
        )

        print("MQLs whose seller state/category matches an identified gap:")
        print(mql_candidates.head(100).to_string(index=False))


In [ ]:
# 21. MQL conversion by lead source / type / behaviour

if "mql" in globals() and "closed_deal" in mql.columns:
    for field in [
        "origin",
        "lead_type",
        "lead_behaviour_profile",
        "business_segment",
        "business_type"
    ]:
        if field in mql.columns:
            summary = (
                mql.groupby(field, dropna=False)
                   .agg(
                       mqls=("mql_id", "nunique"),
                       closed_deals=("closed_deal", "sum")
                   )
            )

            summary["conversion_pct"] = (
                summary["closed_deals"] /
                summary["mqls"].replace(0, np.nan) * 100
            )

            print(f"\n{field.upper()}")
            print(
                summary
                .sort_values("mqls", ascending=False)
                .round(2)
                .to_string()
            )


## Visuals

In [ ]:
# STEP 1: Identify high-demand categories with few/no local sellers

candidate_gaps = gap_analysis[
    (gap_analysis["category_orders"] >= 50) &
    (gap_analysis["local_category_sellers"] <= 1)
].copy()

candidate_gaps = candidate_gaps.sort_values(
    ["local_category_sellers", "category_orders"],
    ascending=[True, False]
)

print("HIGH-DEMAND CATEGORIES WITH FEW/NO LOCAL SELLERS")
print("=" * 80)

print(
    candidate_gaps[
        [
            "customer_state",
            "product_category_name_english",
            "category_orders",
            "local_category_sellers",
            "orders_per_local_seller"
        ]
    ]
    .head(50)
    .to_string(index=False)
)

In [ ]:
# STEP 2: Delivery performance of the identified geographic/category gaps

gap_delivery = candidate_gaps[
    [
        "customer_state",
        "product_category_name_english",
        "category_orders",
        "local_category_sellers"
    ]
].merge(
    outcomes[
        [
            "customer_state",
            "product_category_name_english",
            "avg_delivery_days",
            "median_delivery_days",
            "late_rate",
            "avg_delay_days",
            "same_state_pct",
            "out_of_state_pct",
            "avg_review",
            "low_rating_pct"
        ]
    ],
    on=[
        "customer_state",
        "product_category_name_english"
    ],
    how="left"
)

gap_delivery["late_rate_pct"] = (
    gap_delivery["late_rate"] * 100
).round(2)

print("GEOGRAPHIC/CATEGORY GAPS — DELIVERY & REVIEW PERFORMANCE")
print("=" * 100)

print(
    gap_delivery[
        [
            "customer_state",
            "product_category_name_english",
            "category_orders",
            "local_category_sellers",
            "out_of_state_pct",
            "avg_delivery_days",
            "median_delivery_days",
            "late_rate_pct",
            "avg_delay_days",
            "avg_review",
            "low_rating_pct"
        ]
    ]
    .sort_values(
        ["late_rate_pct", "category_orders"],
        ascending=[False, False]
    )
    .head(50)
    .round(2)
    .to_string(index=False)
)

In [ ]:
# STEP 3: Does local seller coverage affect delivery and reviews?

coverage_test = gap_analysis[
    gap_analysis["category_orders"] >= 50
].copy()

# Group the number of local sellers
coverage_test["seller_coverage"] = pd.cut(
    coverage_test["local_category_sellers"],
    bins=[-1, 0, 1, 3, float("inf")],
    labels=[
        "0 local sellers",
        "1 local seller",
        "2-3 local sellers",
        "4+ local sellers"
    ]
)

coverage_result = (
    coverage_test
    .groupby("seller_coverage", observed=False)
    .agg(
        state_category_combinations=("customer_state", "count"),
        total_orders=("category_orders", "sum"),
        avg_delivery_days=("avg_delivery_days", "mean"),
        avg_late_rate=("late_rate", "mean"),
        avg_review_score=("avg_review", "mean"),
        avg_low_rating_rate=("low_rating_rate", "mean"),
        avg_out_of_state_fulfilment=("out_of_state_pct", "mean")
    )
    .reset_index()
)

coverage_result["late_rate_pct"] = (
    coverage_result["avg_late_rate"] * 100
).round(2)

coverage_result["low_rating_pct"] = (
    coverage_result["avg_low_rating_rate"] * 100
).round(2)

print("LOCAL SELLER COVERAGE VS DELIVERY & CUSTOMER EXPERIENCE")
print("=" * 100)

print(
    coverage_result[
        [
            "seller_coverage",
            "state_category_combinations",
            "total_orders",
            "avg_delivery_days",
            "late_rate_pct",
            "avg_review_score",
            "low_rating_pct",
            "avg_out_of_state_fulfilment"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

In [ ]:
# STEP 4: Rank geographic/category gaps for potential intervention

priority = gap_delivery.copy()

# Create a transparent priority score.
# Higher demand, fewer sellers, higher late rate and lower reviews increase priority.

priority["demand_score"] = (
    priority["category_orders"] /
    priority["category_orders"].max()
)

priority["seller_gap_score"] = np.where(
    priority["local_category_sellers"] == 0,
    1.0,
    1 / (priority["local_category_sellers"] + 1)
)

priority["late_score"] = (
    priority["late_rate_pct"] / 100
)

priority["review_risk_score"] = (
    (5 - priority["avg_review"]) / 4
)

# Combined score.
priority["priority_score"] = (
    0.30 * priority["demand_score"] +
    0.25 * priority["seller_gap_score"] +
    0.25 * priority["late_score"] +
    0.20 * priority["review_risk_score"]
)

# Label the reason for prioritisation
priority["gap_type"] = np.select(
    [
        priority["local_category_sellers"] == 0,
        priority["local_category_sellers"] == 1
    ],
    [
        "No local seller",
        "Only one local seller"
    ],
    default="Low local coverage"
)

priority = priority.sort_values(
    "priority_score",
    ascending=False
)

print("PRIORITY GEOGRAPHIC/CATEGORY GAPS")
print("=" * 120)

print(
    priority[
        [
            "customer_state",
            "product_category_name_english",
            "category_orders",
            "local_category_sellers",
            "out_of_state_pct",
            "avg_delivery_days",
            "late_rate_pct",
            "avg_review",
            "low_rating_pct",
            "gap_type",
            "priority_score"
        ]
    ]
    .head(30)
    .round(2)
    .to_string(index=False)
)

In [ ]:
# STEP 5: Match MQLs to geographic/category supply gaps

# First check whether the MQL table was successfully created
if "mql" not in globals():
    print("MQL table is not available.")
else:

    # We need these fields for a meaningful match
    required_fields = [
        "mql_id",
        "seller_state",
        "seller_city",
        "product_category_name_english"
    ]

    available_fields = [
        c for c in required_fields
        if c in mql.columns
    ]

    print("Available MQL matching fields:")
    print(available_fields)

    if len(available_fields) < 4:

        print(
            "\nCannot perform exact MQL-to-gap matching because "
            "one or more required fields are missing."
        )

    else:

        # Keep only the fields needed for matching
        mql_match = mql[
            [
                "mql_id",
                "seller_state",
                "seller_city",
                "product_category_name_english"
            ]
        ].copy()

        # Match MQL seller state to customer state where a gap exists
        mql_gap_candidates = mql_match.merge(
            priority[
                [
                    "customer_state",
                    "product_category_name_english",
                    "category_orders",
                    "local_category_sellers",
                    "out_of_state_pct",
                    "avg_delivery_days",
                    "late_rate_pct",
                    "avg_review",
                    "low_rating_pct",
                    "priority_score"
                ]
            ],
            left_on=[
                "seller_state",
                "product_category_name_english"
            ],
            right_on=[
                "customer_state",
                "product_category_name_english"
            ],
            how="inner"
        )

        mql_gap_candidates = mql_gap_candidates.sort_values(
            "priority_score",
            ascending=False
        )

        print("\nMQLs MATCHING IDENTIFIED GEOGRAPHIC/CATEGORY GAPS")
        print("=" * 110)

        print(
            mql_gap_candidates[
                [
                    "mql_id",
                    "seller_state",
                    "seller_city",
                    "product_category_name_english",
                    "category_orders",
                    "local_category_sellers",
                    "out_of_state_pct",
                    "avg_delivery_days",
                    "late_rate_pct",
                    "avg_review",
                    "low_rating_pct",
                    "priority_score"
                ]
            ]
            .head(100)
            .round(2)
            .to_string(index=False)
        )

## Visuals

In [ ]:
# Check available geographic columns

geo_columns = [
    col for col in df.columns
    if any(word in col.lower() for word in
           ["lat", "lng", "longitude", "latitude", "zip", "city", "state"])
]

print("Geographic columns available:")
print(geo_columns)

In [ ]:
# STEP 1: Prepare customer and seller geographic concentration

# Customer concentration
customer_geo = (
    df.groupby("customer_state")
      .agg(
          customers=("customer_unique_id", "nunique"),
          customer_orders=("order_id", "nunique")
      )
      .reset_index()
)

# Seller concentration
seller_geo = (
    df.groupby("seller_state")
      .agg(
          sellers=("seller_id", "nunique"),
          seller_orders=("order_id", "nunique")
      )
      .reset_index()
)

# Calculate percentage share
customer_geo["customer_share_pct"] = (
    customer_geo["customers"] /
    customer_geo["customers"].sum() * 100
)

seller_geo["seller_share_pct"] = (
    seller_geo["sellers"] /
    seller_geo["sellers"].sum() * 100
)

print("Customer geographic concentration:")
display(
    customer_geo.sort_values(
        "customer_share_pct",
        ascending=False
    ).round(2)
)

print("\nSeller geographic concentration:")
display(
    seller_geo.sort_values(
        "seller_share_pct",
        ascending=False
    ).round(2)
)

In [ ]:
# STEP 2: Customer vs seller geographic imbalance

geo_balance = customer_geo[
    ["customer_state", "customers", "customer_share_pct"]
].merge(
    seller_geo[
        ["seller_state", "sellers", "seller_share_pct"]
    ],
    left_on="customer_state",
    right_on="seller_state",
    how="outer"
)

geo_balance = geo_balance.drop(columns=["seller_state"])

geo_balance["customers"] = geo_balance["customers"].fillna(0)
geo_balance["sellers"] = geo_balance["sellers"].fillna(0)
geo_balance["customer_share_pct"] = (
    geo_balance["customer_share_pct"].fillna(0)
)
geo_balance["seller_share_pct"] = (
    geo_balance["seller_share_pct"].fillna(0)
)

# Positive = customer share is greater than seller share
geo_balance["customer_minus_seller_pct"] = (
    geo_balance["customer_share_pct"]
    - geo_balance["seller_share_pct"]
)

# Customers per seller
geo_balance["customers_per_seller"] = np.where(
    geo_balance["sellers"] > 0,
    geo_balance["customers"] / geo_balance["sellers"],
    np.inf
)

geo_balance = geo_balance.sort_values(
    "customer_minus_seller_pct",
    ascending=False
)

print("CUSTOMER VS SELLER GEOGRAPHIC IMBALANCE")
print("=" * 80)

display(
    geo_balance[
        [
            "customer_state",
            "customers",
            "customer_share_pct",
            "sellers",
            "seller_share_pct",
            "customer_minus_seller_pct",
            "customers_per_seller"
        ]
    ].round(2)
)

In [ ]:
import plotly.express as px

# Unique customer locations
customer_locations = (
    df[
        [
            "customer_zip_code_prefix",
            "customer_geolocation_lat",
            "customer_geolocation_lng",
            "customer_state"
        ]
    ]
    .drop_duplicates()
    .dropna()
)

fig = px.scatter_map(
    customer_locations,
    lat="customer_geolocation_lat",
    lon="customer_geolocation_lng",
    zoom=3.2,
    center={"lat": -14.2, "lon": -51.9},
    map_style="open-street-map",
    title="Geographic Concentration of Customers",
    hover_data={
        "customer_state": True,
        "customer_zip_code_prefix": True,
        "customer_geolocation_lat": False,
        "customer_geolocation_lng": False
    }
)

# Single colour for all points
fig.update_traces(
    marker=dict(
        size=5,
        opacity=0.45
    )
)

fig.update_layout(
    height=700
)

fig.show()

In [ ]:
# Unique seller locations
seller_locations = (
    df[
        [
            "seller_zip_code_prefix",
            "seller_geolocation_lat",
            "seller_geolocation_lng",
            "seller_state"
        ]
    ]
    .drop_duplicates()
    .dropna()
)

fig = px.scatter_map(
    seller_locations,
    lat="seller_geolocation_lat",
    lon="seller_geolocation_lng",
    zoom=3.2,
    center={"lat": -14.2, "lon": -51.9},
    map_style="open-street-map",
    title="Geographic Concentration of Sellers",
    hover_data={
        "seller_state": True,
        "seller_zip_code_prefix": True,
        "seller_geolocation_lat": False,
        "seller_geolocation_lng": False
    }
)

fig.update_traces(
    marker=dict(
        size=6,
        opacity=0.55
    )
)

fig.update_layout(
    height=700
)

fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------------------------------------
# Prepare unique customer and seller locations
# ---------------------------------------------------------

customer_locations = (
    df[
        [
            "customer_zip_code_prefix",
            "customer_geolocation_lat",
            "customer_geolocation_lng",
            "customer_state"
        ]
    ]
    .drop_duplicates()
    .dropna()
)

seller_locations = (
    df[
        [
            "seller_zip_code_prefix",
            "seller_geolocation_lat",
            "seller_geolocation_lng",
            "seller_state"
        ]
    ]
    .drop_duplicates()
    .dropna()
)

# ---------------------------------------------------------
# Create side-by-side maps
# ---------------------------------------------------------

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "map"}, {"type": "map"}]],
    subplot_titles=(
        "Customer Geographic Concentration",
        "Seller Geographic Concentration"
    )
)

# ---------------------------------------------------------
# Customer map
# ---------------------------------------------------------

fig.add_trace(
    go.Scattermap(
        lat=customer_locations["customer_geolocation_lat"],
        lon=customer_locations["customer_geolocation_lng"],
        mode="markers",
        marker=dict(
            size=5,
            opacity=0.45
        ),
        text=customer_locations["customer_state"],
        hovertemplate=(
            "<b>Customer</b><br>"
            "State: %{text}<br>"
            "Latitude: %{lat:.4f}<br>"
            "Longitude: %{lon:.4f}"
            "<extra></extra>"
        ),
        name="Customers"
    ),
    row=1,
    col=1
)

# ---------------------------------------------------------
# Seller map
# ---------------------------------------------------------

fig.add_trace(
    go.Scattermap(
        lat=seller_locations["seller_geolocation_lat"],
        lon=seller_locations["seller_geolocation_lng"],
        mode="markers",
        marker=dict(
            size=7,
            opacity=0.60
        ),
        text=seller_locations["seller_state"],
        hovertemplate=(
            "<b>Seller</b><br>"
            "State: %{text}<br>"
            "Latitude: %{lat:.4f}<br>"
            "Longitude: %{lon:.4f}"
            "<extra></extra>"
        ),
        name="Sellers"
    ),
    row=1,
    col=2
)

# ---------------------------------------------------------
# Map configuration
# ---------------------------------------------------------

fig.update_layout(
    map=dict(
        style="open-street-map",
        center=dict(lat=-14.2, lon=-51.9),
        zoom=3.2
    ),
    map2=dict(
        style="open-street-map",
        center=dict(lat=-14.2, lon=-51.9),
        zoom=3.2
    ),

    title="Geographical Concentration: Customers vs Sellers",

    height=650,

    showlegend=False,

    margin=dict(
        l=10,
        r=10,
        t=70,
        b=10
    )
)

fig.show()

In [ ]:
import plotly.express as px

# ---------------------------------------------------------
# Prepare data for supply-gap heatmap
# ---------------------------------------------------------

heatmap_data = gap_analysis.copy()

# Keep categories with meaningful demand
heatmap_data = heatmap_data[
    heatmap_data["category_orders"] >= 50
].copy()

# Create readable labels
heatmap_data["category_label"] = (
    heatmap_data["product_category_name_english"]
    .str.replace("_", " ")
    .str.title()
)

# Pivot: states x categories
heatmap_matrix = heatmap_data.pivot_table(
    index="customer_state",
    columns="category_label",
    values="local_category_sellers",
    aggfunc="sum",
    fill_value=0
)

fig = px.imshow(
    heatmap_matrix,
    labels=dict(
        x="Product Category",
        y="Customer State",
        color="Local Sellers"
    ),
    title="Local Seller Availability by Customer State and Product Category",
    aspect="auto",
    text_auto=True
)

fig.update_layout(
    height=750,
    xaxis_title="Product Category",
    yaxis_title="Customer State"
)

fig.show()

In [ ]:
import plotly.express as px

# ---------------------------------------------------------
# Prepare priority-gap data
# ---------------------------------------------------------

plot_data = priority.copy()

# Focus on meaningful demand
plot_data = plot_data[
    plot_data["category_orders"] >= 50
].copy()

# Take the top 20 opportunities
plot_data = (
    plot_data
    .sort_values("priority_score", ascending=False)
    .head(20)
    .sort_values("category_orders", ascending=True)
)

# Create state + category label
plot_data["opportunity"] = (
    plot_data["customer_state"].str.upper()
    + " — "
    + plot_data["product_category_name_english"]
    .str.replace("_", " ")
    .str.title()
)

# Seller label
plot_data["seller_label"] = (
    plot_data["local_category_sellers"]
    .astype(int)
    .astype(str)
    + " local seller"
)

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

fig = px.bar(
    plot_data,
    x="category_orders",
    y="opportunity",
    orientation="h",
    text="category_orders",
    hover_data={
        "category_orders": True,
        "local_category_sellers": True,
        "out_of_state_pct": ":.1f",
        "avg_delivery_days": ":.1f",
        "late_rate_pct": ":.1f",
        "avg_review": ":.2f",
        "low_rating_pct": ":.1f",
        "priority_score": ":.2f"
    },
    title="Highest-Priority Geographic & Category Seller Gaps",
    labels={
        "category_orders": "Customer Orders",
        "opportunity": "Customer State — Product Category"
    }
)

fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    height=750,
    xaxis_title="Customer Demand (Orders)",
    yaxis_title="",
    showlegend=False,
    margin=dict(
        l=180,
        r=100,
        t=80,
        b=60
    )
)

fig.show()